In [ ]:
# Einen oder zwei Modellordner unter walinet/models auswählen:
MODELS = ["UNet_ablation_parameter_matched_fixed_traingset_Seed_3", "UNet_ablation_parameter_matched_Seed_3"]
# Für nur ein Modell z.B.: MODELS = ["YNet_ablation"]
LOG_Y = True  # False für lineare Y-Achse
VALIDATION_ONLY = False  # True zeigt ausschließlich Validation Loss

In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt

project_root = Path("..").resolve()
if not (project_root / "models").is_dir():
    project_root = Path(".").resolve()
pattern = re.compile(
    r"Epoch:\s*(\d+),\s*Loss:\s*([0-9.eE+-]+),\s*"
    r"Val Loss:\s*([0-9.eE+-]+)"
)
if not 1 <= len(MODELS) <= 2:
    raise ValueError("MODELS must contain one or two model names.")

plt.figure(figsize=(9, 5))
for model_index, model in enumerate(MODELS):
    loss_path = project_root / "models" / model / "loss.txt"
    if not loss_path.is_file():
        raise FileNotFoundError(loss_path)
    rows = [pattern.search(line) for line in loss_path.read_text().splitlines()]
    rows = [match for match in rows if match is not None]
    if not rows:
        raise ValueError(f"No loss entries found in {loss_path}")
    epochs = [int(match.group(1)) for match in rows]
    training_loss = [float(match.group(2)) for match in rows]
    validation_loss = [float(match.group(3)) for match in rows]
    color = f"C{model_index}"
    if not VALIDATION_ONLY:
        plt.plot(epochs, training_loss, color=color,
                 label=f"{model} – Training")
    plt.plot(epochs, validation_loss, linestyle="--", color=color,
             label=f"{model} – Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
if LOG_Y:
    plt.yscale("log")
plt.title("Training curves")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()